# Audit two ACS home-language estimates

Follow a denominator, a language subgroup and their published margins of error from original Census fields to a transparent calculation. This notebook uses **2020–2024 ACS five-year estimates**, table C16001, for **ZCTA 78201** and **Bexar County, Texas (48029)**. These are two separate worked records, not a comparison or a combined service area.

**Reader path:** verify the bundled sources → identify each geography → inspect count fields and source margins → calculate a share using the same geography → exercise safeguards. Every code cell uses the Python standard library. Saved outputs are from an actual clean sequential run; rerun with `python reproduce.py` from a downloaded copy of this folder. No network, account or data upload is needed.

The result is an audit trail, not a count of a church's visitors, an individual's language preference or evidence of translation accuracy.


## 1. Establish the snapshot

The bundled `.dat` excerpt keeps the original header and **all 77 fields** of the two selected Census rows. The table-shell excerpt supplies the universe and 38 line labels; the county gazetteer supplies the county name. `provenance.json` records the source URLs, complete-file hashes, excerpt hashes, selected line numbers and actual retrieval times. Large original downloads are excluded from this publication.

The survey period is **January 2020 through December 2024**, not a point-in-time 2024 census. Retrieval time and notebook execution time are separate metadata. [Census explains five-year estimates](https://www.census.gov/programs-surveys/acs/guidance/estimates.html).


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import re
from decimal import Decimal, ROUND_HALF_UP

BASE = Path.cwd()
provenance = json.loads((BASE / "provenance.json").read_text(encoding="utf-8"))
for item in provenance["files"]:
    content = (BASE / item["path"]).read_bytes()
    if hashlib.sha256(content).hexdigest() != item["sha256"]:
        raise ValueError("Bundled source changed: " + item["path"])
    print("Hash verified:", item["path"])
print("Survey period:", provenance["survey_period"])
print("Source retrieved (UTC):", provenance["files"][0]["retrieved_at"])
print("Table universe:", provenance["universe"])

def read_table(name, delimiter):
    with (BASE / "data" / name).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream, delimiter=delimiter)
        result = list(reader)
        if any(None in row or any(v is None for v in row.values()) for row in result):
            raise ValueError("Unexpected row width: " + name)
        return result

raw_rows = read_table("c16001-two-geographies.dat", "|")
if len(raw_rows) != 2 or len(raw_rows[0]) != 77:
    raise ValueError("Expected two rows, each with GEO_ID plus 38 estimate/MOE pairs")
by_geo = {row["GEO_ID"]: row for row in raw_rows}
expected_geo = {"0500000US48029", "860Z200US78201"}
if set(by_geo) != expected_geo or len(by_geo) != len(raw_rows):
    raise ValueError("Missing, duplicate or unexpected geography")
print("Original geography rows:", len(raw_rows), "| Fields per row:", len(raw_rows[0]))


Hash verified: data/c16001-two-geographies.dat
Hash verified: data/c16001-table-shell.txt
Hash verified: data/county-48029.tsv
Survey period: 2020-2024
Source retrieved (UTC): 2026-10-28T08:33:49.727717+00:00
Table universe: Population 5 years and over
Original geography rows: 2 | Fields per row: 77


## 2. Keep geographic identities intact

`0500000US48029` is the county record; `860Z200US78201` is the ZCTA record. A [ZCTA is a Census statistical representation](https://www.census.gov/programs-surveys/geography/guidance/geo-areas/zctas.html), not a USPS delivery route. The shared five-digit length does not make county FIPS and ZCTA codes interchangeable.

No crosswalk or containment calculation is included. Do not sum these areas, subtract one from the other, or treat their difference as statistically significant. The code refuses a share with a numerator and denominator from different geography IDs.


In [2]:
gazetteer = read_table("county-48029.tsv", "\t")
if len(gazetteer) != 1 or gazetteer[0]["GEOID"] != "48029":
    raise ValueError("County gazetteer identity mismatch")
if gazetteer[0]["USPS"] != "TX" or gazetteer[0]["NAME"] != "Bexar County":
    raise ValueError("Unexpected county label")
identities = {
    "0500000US48029": "Bexar County, Texas | county FIPS 48029",
    "860Z200US78201": "ZCTA 78201 | Census statistical area",
}
for geo_id, label in identities.items():
    print(geo_id)
    print("  " + label)
print("No relationship between these boundaries is inferred.")


0500000US48029
  Bexar County, Texas | county FIPS 48029
860Z200US78201
  ZCTA 78201 | Census statistical area
No relationship between these boundaries is inferred.


## 3. Read the denominator and subgroup hierarchy

This table covers **people age five and over**, not everyone in an area. The Spanish line counts the home-language group; it is not a count of people who require translation. Its two indented child lines describe reported English-speaking ability.

The summary-file spelling `C16001_E003` corresponds to API variable `C16001_003E`; `M` marks the published margin. The [official table metadata](https://api.census.gov/data/2024/acs/acs5/groups/C16001.html) and bundled table shell define the fields. A margin is in **people**, just like its count estimate. Census publishes these margins at the **90% confidence level** ([2020–2024 accuracy guidance](https://www2.census.gov/programs-surveys/acs/tech_docs/accuracy/MultiyearACSAccuracyofData2024.pdf)).


In [3]:
shell = read_table("c16001-table-shell.txt", "|")
labels = {row["Unique ID"]: row for row in shell}
if len(labels) != 38 or any(row["Universe"] != "Population 5 years and over" for row in shell):
    raise ValueError("Unexpected C16001 table shell")
field_lines = (1, 2, 3, 4, 5)
for line in field_lines:
    item = labels[f"C16001_{line:03d}"]
    indent = "  " * int(item["Indent"])
    print(f"{line:03d} {indent}{item['Label']}")
print("E = count estimate; M = its published count margin of error.")


001 Total:
002   Speak only English
003   Spanish:
004     Speak English "very well"
005     Speak English less than "very well"
E = count estimate; M = its published count margin of error.


## 4. Preserve missing values before calculating

ACS can publish annotations or special negative codes instead of numeric counts or margins. [Census documents these values](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html). This deliberately narrow parser accepts only nonnegative whole-number values for this count table. It preserves the original string and a reason for everything else. It does **not** replace missing values with zero or claim that all negative ACS variables are invalid—other tables can contain meaningful negative monetary values.

The two genuine source rows are shown below. Synthetic sentinel examples appear later, clearly separate from source observations.


In [4]:
def count_value(raw):
    text = "" if raw is None else str(raw)
    stripped = text.strip()
    if not stripped:
        return {"raw": text, "value": None, "state": "missing/blank"}
    if re.fullmatch(r"-[0-9]+", stripped):
        return {"raw": text, "value": None, "state": "negative code; retained, not a count"}
    if not re.fullmatch(r"[0-9]+", stripped):
        return {"raw": text, "value": None, "state": "annotation or unexpected value; retained"}
    return {"raw": text, "value": int(stripped), "state": "numeric"}

def field(row, line, measure="E"):
    if measure not in ("E", "M"):
        raise ValueError("Use E for estimate or M for published count margin")
    return count_value(row[f"C16001_{measure}{line:03d}"])

def shown(value):
    if value["value"] is None:
        return repr(value["raw"]) + " [" + value["state"] + "]"
    return f"{value['value']:,}"

for geo_id, row in by_geo.items():
    print("\n" + identities[geo_id])
    for line in field_lines:
        label = labels[f"C16001_{line:03d}"]["Label"]
        print(f"  {line:03d} {label}")
        print("      Estimate:", shown(field(row, line)), "people")
        print("      Source MOE:", shown(field(row, line, "M")), "people (90%)")

source_issues = [(r["GEO_ID"], key, count_value(value)["state"], value)
                 for r in raw_rows for key, value in r.items()
                 if key != "GEO_ID" and count_value(value)["value"] is None]
print("\nNon-numeric or missing values across all 152 source measures:", len(source_issues))
for issue in source_issues:
    print(issue)



Bexar County, Texas | county FIPS 48029
  001 Total:
      Estimate: 1,934,539 people
      Source MOE: 69 people (90%)
  002 Speak only English
      Estimate: 1,217,847 people
      Source MOE: 8,835 people (90%)
  003 Spanish:
      Estimate: 634,387 people
      Source MOE: 7,392 people (90%)
  004 Speak English "very well"
      Estimate: 441,069 people
      Source MOE: 5,908 people (90%)
  005 Speak English less than "very well"
      Estimate: 193,318 people
      Source MOE: 5,657 people (90%)

ZCTA 78201 | Census statistical area
  001 Total:
      Estimate: 40,357 people
      Source MOE: 2,667 people (90%)
  002 Speak only English
      Estimate: 17,476 people
      Source MOE: 1,813 people (90%)
  003 Spanish:
      Estimate: 22,100 people
      Source MOE: 1,920 people (90%)
  004 Speak English "very well"
      Estimate: 14,265 people
      Source MOE: 1,410 people (90%)
  005 Speak English less than "very well"
      Estimate: 7,835 people
      Source MOE: 916 people 

## 5. Calculate one point-estimate share per geography

The numerator is the Spanish home-language estimate (line 003). The denominator is the age-five-and-over total (line 001), from the **same** row. The formula is `100 × numerator / denominator`, rounded to two decimal places. A second share uses the Spanish subgroup reporting English less than “very well” (line 005), divided by the Spanish total (line 003). Its denominator is explicitly different.

These are descriptive **point estimates only**. This notebook derives **no percentage margin of error, confidence interval, ranking or significance test**. Dividing a count margin by the denominator would not account for the uncertainty in a ratio. The original margins stay alongside their original counts, not beside a calculated percentage.


In [5]:
def point_share(numerator, denominator, numerator_geo, denominator_geo):
    if numerator_geo != denominator_geo:
        raise ValueError("Different geography IDs: a share requires the same source row")
    if numerator is None or denominator is None:
        return None
    if type(numerator) is not int or type(denominator) is not int:
        raise ValueError("Point shares require parsed integer counts")
    if denominator <= 0 or numerator < 0 or numerator > denominator:
        raise ValueError("Counts do not define a valid part/whole share")
    return (Decimal(numerator) * 100 / Decimal(denominator)).quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP)

def percent_text(value):
    return "not calculated: a source count is missing" if value is None else f"{value}% (point estimate only)"

audit = []
for geo_id, row in by_geo.items():
    total = field(row, 1)["value"]
    spanish = field(row, 3)["value"]
    very_well = field(row, 4)["value"]
    less_very_well = field(row, 5)["value"]
    share_total = point_share(spanish, total, geo_id, geo_id)
    share_subgroup = point_share(less_very_well, spanish, geo_id, geo_id)
    child_sum = None if None in (spanish, very_well, less_very_well) else very_well + less_very_well == spanish
    audit.append({"geo_id":geo_id, "spanish_share_age5plus":str(share_total),
                  "less_very_well_share_spanish":str(share_subgroup), "children_sum_to_parent":child_sum})
    print("\n" + identities[geo_id])
    print("  Spanish / age 5+ total:", spanish, "/", total)
    print("   ", percent_text(share_total))
    print("  English less than very well / Spanish group:", less_very_well, "/", spanish)
    print("   ", percent_text(share_subgroup))
    print("  Spanish children sum to their parent in this snapshot:", child_sum)
print("\nNo percentage MOE or cross-geography comparison was calculated.")



Bexar County, Texas | county FIPS 48029
  Spanish / age 5+ total: 634387 / 1934539
    32.79% (point estimate only)
  English less than very well / Spanish group: 193318 / 634387
    30.47% (point estimate only)
  Spanish children sum to their parent in this snapshot: True

ZCTA 78201 | Census statistical area
  Spanish / age 5+ total: 22100 / 40357
    54.76% (point estimate only)
  English less than very well / Spanish group: 7835 / 22100
    35.45% (point estimate only)
  Spanish children sum to their parent in this snapshot: True

No percentage MOE or cross-geography comparison was calculated.


## 6. Exercise the failure cases

These **synthetic checks are not Census observations**. They make a sentinel, a zero and an incompatible denominator visibly behave differently. A zero count remains a real zero; a missing count does not become zero. A source margin can be missing while its estimate remains usable as a point estimate.


In [6]:
synthetic = ["0", "", "-666666666", "-555555555", "*****", "N", "12.5"]
for value in synthetic:
    parsed = count_value(value)
    print(repr(value), "->", parsed["value"], "|", parsed["state"])
assert count_value("0")["value"] == 0
assert all(count_value(v)["value"] is None for v in synthetic[1:])
assert point_share(None, 100, "same", "same") is None
assert point_share(0, 100, "same", "same") == Decimal("0.00")
assert count_value("25")["value"] == 25 and count_value("-555555555")["value"] is None

guards = [
    ("different geographies", (22100, 1934539, "860Z200US78201", "0500000US48029")),
    ("zero denominator", (0, 0, "same", "same")),
    ("part greater than whole", (101, 100, "same", "same")),
    ("negative value", (-1, 100, "same", "same")),
]
for description, args in guards:
    try:
        point_share(*args)
    except ValueError as error:
        print("Guard passed:", description, "|", error)
    else:
        raise AssertionError("Guard failed: " + description)
print("Synthetic safeguards passed; original data were not modified.")


'0' -> 0 | numeric
'' -> None | missing/blank
'-666666666' -> None | negative code; retained, not a count
'-555555555' -> None | negative code; retained, not a count
'*****' -> None | annotation or unexpected value; retained
'N' -> None | annotation or unexpected value; retained
'12.5' -> None | annotation or unexpected value; retained
Guard passed: different geographies | Different geography IDs: a share requires the same source row
Guard passed: zero denominator | Counts do not define a valid part/whole share
Guard passed: part greater than whole | Counts do not define a valid part/whole share
Guard passed: negative value | Counts do not define a valid part/whole share
Synthetic safeguards passed; original data were not modified.


## What this audit supports

You can trace every displayed source estimate and count margin to an exact bundled row, verify the five-and-over denominator, and reproduce the two within-row shares. The child-sum check is an arithmetic observation about these records, not proof that the survey is error-free. Published margins describe sampling uncertainty; they do not remove nonsampling error.

Area estimates cannot establish an individual's preference, a congregation's composition, translation demand, causality or product accuracy. The county and ZCTA need not share boundaries. Consult Census methodology before comparing estimates, combining geographies or calculating derived uncertainty.

For an optional interactive way to explore these area-level groups, see [PewTranslate's languages-spoken-by-ZIP-code lookup](https://pewtranslate.com/en/guides/languages-spoken-by-zip-code). That page is an interface/reference, **not the statistics source for this notebook**. The underlying statistics here are directly retrieved Census material.

Original notebook and AI-assisted code by [PewTranslate](https://pewtranslate.com/). Live-translation accounts are closed. This audit does not translate or imply a working production translation service, professional certification or Census endorsement.

### Reproduce or obtain a fresh snapshot

`python reproduce.py` runs every code cell unchanged in a new Python process, blocks network sockets, and compares the generated outputs with the saved outputs. It does not require Jupyter or write to the source files. In an existing Jupyter environment, select a Python 3 kernel and run all cells with this folder as the working directory.

For a **separate, optional network operation**, download the three official source files listed in `provenance.json` into a new folder. Use the recorded row selectors and original headers to create a new snapshot; retain retrieval times and new hashes. Do not silently overwrite this dated evidence or assume that a fresh source hash must match. The [Census table-based summary-file guide](https://www.census.gov/programs-surveys/acs/data/summary-file.html) describes the format. A new snapshot requires rerunning and reviewing the audit, not just changing its date.

Original code and explanatory text: MIT, copyright 2026 szj2ys. Public Census source material retains its attribution; it is not relicensed as original code.
